# BQ2 - Phân tích redeem coupon

Mục tiêu phân tích:
1. Tỷ lệ % household ở mỗi campaign đã redeem coupon
2. Demographic của nhóm household redeem coupon
3. Trong lúc campaign diễn ra, so sánh hành vi mua sắm giữa nhóm household có redeem và không redeem coupon
   (số invoice, số món trung bình/invoice, chi tiêu trung bình)

File này độc lập với `bq1.ipynb` (khác câu hỏi phân tích - BQ1 tập trung vào tác động của campaign lên chi
tiêu, BQ2 tập trung vào hành vi redeem coupon). Chỉ cần 6 file CSV gốc trong cùng folder
(`campaign_desc.csv`, `campaign_table.csv`, `coupon.csv`, `coupon_redempt.csv`, `hh_demographic.csv`,
`transaction_data.csv`) là chạy được, không phụ thuộc vào biến nào từ bq1.ipynb.

In [ ]:
import pandas as pd

campaign_desc = pd.read_csv("campaign_desc.csv")
campaign_table = pd.read_csv("campaign_table.csv")
coupon_redempt = pd.read_csv("coupon_redempt.csv")
hh_demographic = pd.read_csv("hh_demographic.csv")
transaction_data = pd.read_csv("transaction_data.csv")

## Output A - % household mỗi campaign đã redeem coupon

Với mỗi campaign: số household được target (campaign_table) so với số household THỰC SỰ có ít nhất 1 lần
redeem coupon cho campaign đó (coupon_redempt, cột CAMPAIGN có sẵn - không cần join qua coupon.csv).

In [ ]:
targeted_per_campaign = campaign_table.groupby("CAMPAIGN")["household_key"].nunique().rename("N_TARGETED")
redeemed_per_campaign = coupon_redempt.groupby("CAMPAIGN")["household_key"].nunique().rename("N_REDEEMED")

redeem_summary = pd.concat([targeted_per_campaign, redeemed_per_campaign], axis=1).fillna(0)
redeem_summary["N_REDEEMED"] = redeem_summary["N_REDEEMED"].astype(int)
redeem_summary["PCT_REDEEMED"] = (redeem_summary["N_REDEEMED"] / redeem_summary["N_TARGETED"] * 100).round(1)
redeem_summary = redeem_summary.sort_values("PCT_REDEEMED", ascending=False)

print(f"{len(redeem_summary)} campaigns in the table (should be 30 - none dropped)")
display(redeem_summary)

### Insights: Output A

- Tỷ lệ redeem chênh lệch khá nhiều giữa các campaign: cao nhất là campaign 18 (18.9%) và campaign 13
  (18.2%) - cả 2 đều là campaign lớn (hơn 1000 household), trong khi thấp nhất là campaign 6 (1.5%) và
  campaign 11 (2.8%).
- Tính trên toàn bộ: 434/1584 household (27.4%) từng redeem ít nhất 1 coupon (bất kỳ campaign nào).
- Không thấy xu hướng rõ ràng là campaign lớn thì tỷ lệ redeem cao hơn (ví dụ campaign 3 chỉ có 12
  household nhưng tỷ lệ redeem 16.7%, cao hơn nhiều campaign lớn khác).

## Output B - Demographic của nhóm household đã redeem coupon

So sánh demographic giữa 2 nhóm (trong phạm vi household ĐÃ TỪNG được target bởi ít nhất 1 campaign):
"Redeemed" (từng redeem ít nhất 1 coupon) so với "Targeted, no redeem" (được target nhưng chưa bao giờ
redeem).

In [ ]:
redeemed_hh = set(coupon_redempt["household_key"].unique())
all_targeted_hh = set(campaign_table["household_key"].unique())

demo = hh_demographic.copy()
demo["REDEEM_STATUS"] = demo["household_key"].apply(
    lambda h: "Redeemed" if h in redeemed_hh else ("Targeted, no redeem" if h in all_targeted_hh else "Not targeted")
)
demo_scope = demo[demo["household_key"].isin(all_targeted_hh)]  # chỉ xét household đã từng được target

coverage = demo_scope["AGE_DESC"].notna().mean()
print(f"Coverage (tỷ lệ có dữ liệu demographic): {coverage:.1%}")

In [ ]:
# Tỷ lệ độ tuổi: Redeemed so với Targeted-no-redeem
pivot_age_desc = (
    demo_scope.groupby("REDEEM_STATUS")["AGE_DESC"]
    .value_counts(normalize=True)
    .rename("PERCENTAGE")
    .reset_index()
    .pivot(index="REDEEM_STATUS", columns="AGE_DESC", values="PERCENTAGE")
    .fillna(0) * 100
)
display(pivot_age_desc.round(1))

In [ ]:
# Tỷ lệ thu nhập: Redeemed so với Targeted-no-redeem
pivot_income_desc = (
    demo_scope.groupby("REDEEM_STATUS")["INCOME_DESC"]
    .value_counts(normalize=True)
    .rename("PERCENTAGE")
    .reset_index()
    .pivot(index="REDEEM_STATUS", columns="INCOME_DESC", values="PERCENTAGE")
    .fillna(0) * 100
)
display(pivot_income_desc.round(1))

In [ ]:
# Tỷ lệ quy mô hộ: Redeemed so với Targeted-no-redeem
pivot_household_size_desc = (
    demo_scope.groupby("REDEEM_STATUS")["HOUSEHOLD_SIZE_DESC"]
    .value_counts(normalize=True)
    .rename("PERCENTAGE")
    .reset_index()
    .pivot(index="REDEEM_STATUS", columns="HOUSEHOLD_SIZE_DESC", values="PERCENTAGE")
    .fillna(0) * 100
)
display(pivot_household_size_desc.round(1))

In [ ]:
# Tỷ lệ tình trạng hôn nhân: Redeemed so với Targeted-no-redeem
pivot_marital_status_code = (
    demo_scope.groupby("REDEEM_STATUS")["MARITAL_STATUS_CODE"]
    .value_counts(normalize=True)
    .rename("PERCENTAGE")
    .reset_index()
    .pivot(index="REDEEM_STATUS", columns="MARITAL_STATUS_CODE", values="PERCENTAGE")
    .fillna(0) * 100
)
display(pivot_marital_status_code.round(1))

In [ ]:
# Tỷ lệ chủ nhà: Redeemed so với Targeted-no-redeem
pivot_homeowner_desc = (
    demo_scope.groupby("REDEEM_STATUS")["HOMEOWNER_DESC"]
    .value_counts(normalize=True)
    .rename("PERCENTAGE")
    .reset_index()
    .pivot(index="REDEEM_STATUS", columns="HOMEOWNER_DESC", values="PERCENTAGE")
    .fillna(0) * 100
)
display(pivot_homeowner_desc.round(1))

In [ ]:
# Tỷ lệ nhóm có con: Redeemed so với Targeted-no-redeem
pivot_kid_category_desc = (
    demo_scope.groupby("REDEEM_STATUS")["KID_CATEGORY_DESC"]
    .value_counts(normalize=True)
    .rename("PERCENTAGE")
    .reset_index()
    .pivot(index="REDEEM_STATUS", columns="KID_CATEGORY_DESC", values="PERCENTAGE")
    .fillna(0) * 100
)
display(pivot_kid_category_desc.round(1))

### Insights: Output B

- **Nhóm redeem lệch về độ tuổi lớn hơn một chút**: nhóm 45-54 tuổi chiếm 39.5% ở nhóm Redeemed so với
  33.0% ở nhóm không redeem; ngược lại nhóm 19-24 tuổi chỉ chiếm 2.9% (Redeemed) so với 8.0% (không
  redeem) - người trẻ dùng coupon ít hơn hẳn.
- **Nhóm redeem có thu nhập trung bình-khá cao hơn một chút, nhưng không theo quy luật tuyến tính**: nhóm
  thu nhập 50-74K chiếm 28.6% (Redeemed) so với 21.8% (không redeem) - đông nhất trong nhóm Redeemed;
  nhóm thu nhập thấp (15-24K: 5.8% so với 11.1%, dưới 15K: 7.1% so với 8.2%) redeem ít hơn hẳn. Đáng chú
  ý là nhóm thu nhập cao nhất (250K+) lại chiếm tỷ lệ THẤP hơn (0.6% so với 1.8%) - không phải cứ thu
  nhập cao là redeem nhiều, mà là nhóm TRUNG BÌNH-KHÁ (50-74K, 150-199K) mới hay dùng coupon.
- **Quy mô hộ gia đình**: khá giống nhau, nhưng hộ 4 người (8.0% so với 5.1%) và hộ từ 5 người trở lên
  (9.6% so với 6.0%) có xu hướng redeem nhiều hơn hộ 1 người (30.5% so với 34.1%) - hộ đông người hơn một
  chút thì hay dùng coupon hơn.
- **Tình trạng hôn nhân**: nhóm A (đã kết hôn) chiếm 47.3% (Redeemed) so với 37.9% (không redeem) - redeem
  nhiều hơn; ngược lại nhóm U (chưa rõ) 38.3% so với 46.8% - redeem ít hơn. Nhóm B (độc thân) gần như
  không đổi (14.5% so với 15.4%).
- **Chủ nhà - chênh lệch rõ nhất trong tất cả các biến demographic**: nhóm chủ nhà (Homeowner) chiếm
  70.1% ở nhóm Redeemed nhưng chỉ 57.0% ở nhóm không redeem; ngược lại nhóm chưa rõ (Unknown) 20.6% so
  với 35.4%. Người sở hữu nhà dùng coupon nhiều hơn người thuê nhà/chưa rõ tình trạng nhà ở một cách rõ
  rệt.
- **Nhóm có con**: khá giống nhau, nhóm "chưa có con/chưa rõ" chiếm đa số ở cả 2 nhóm (67.5% so với
  73.1%); hộ có 2 con (9.0% so với 6.0%) và từ 3 con trở lên (10.0% so với 6.5%) nhỉnh hơn một chút ở
  nhóm Redeemed.

**Tóm lại**: người dùng coupon có xu hướng lớn tuổi hơn (nhưng không phải già nhất), thu nhập trung
bình-khá (không phải cao nhất), đã kết hôn, hộ đông người hơn một chút - và đặc biệt là **chủ nhà
(homeowner)** là yếu tố chênh lệch rõ rệt nhất trong tất cả các biến demographic đã xem.

## Output C - Hành vi mua sắm trong lúc campaign: redeem so với không redeem coupon

Với mỗi cặp (household, campaign) mà household đó được target, xác định household có redeem coupon CHO
CHÍNH campaign đó hay không (một household có thể redeem ở campaign này nhưng không redeem ở campaign
khác). Trong khung ngày campaign, tính: số invoice (trip), số món trung bình/invoice, chi tiêu trung bình
- rồi gộp lại (pooled) theo 2 nhóm Redeemed / Not redeemed trên toàn bộ 30 campaign.

**Bước 1 - Gắn cờ `REDEEMED` cho từng cặp (household, campaign):** household đó có redeem coupon của đúng campaign này hay không.

In [ ]:
hh_campaign = campaign_table[["household_key", "CAMPAIGN"]].drop_duplicates().merge(
    campaign_desc[["CAMPAIGN", "START_DAY", "END_DAY"]].drop_duplicates(subset="CAMPAIGN"), on="CAMPAIGN", how="left"
)
redeem_pairs = set(zip(coupon_redempt["household_key"], coupon_redempt["CAMPAIGN"]))
hh_campaign["REDEEMED"] = list(zip(hh_campaign["household_key"], hh_campaign["CAMPAIGN"]))
hh_campaign["REDEEMED"] = hh_campaign["REDEEMED"].isin(redeem_pairs)
hh_campaign.head()

**Bước 2 - Với mỗi campaign, tính hành vi mua sắm (`n_invoices`, avg items/invoice, tổng chi tiêu) cho từng household trong đúng khung ngày của campaign đó, rồi gộp tất cả lại thành 1 bảng.**

In [ ]:
per_household_frames = []
for camp, g in hh_campaign.groupby("CAMPAIGN"):
    start, end = g["START_DAY"].iloc[0], g["END_DAY"].iloc[0]
    hh_list = g["household_key"].values
    redeemed_flag = g.set_index("household_key")["REDEEMED"]

    sub = transaction_data[
        transaction_data["household_key"].isin(hh_list) & transaction_data["DAY"].between(start, end)
    ]
    if sub.empty:
        continue
    per_hh = sub.groupby("household_key").agg(
        n_invoices=("BASKET_ID", "nunique"),
        total_spend=("SALES_VALUE", "sum"),
        total_items=("QUANTITY", "sum"),
    )
    per_hh["avg_items_per_invoice"] = per_hh["total_items"] / per_hh["n_invoices"]
    per_hh["CAMPAIGN"] = camp
    per_hh["REDEEMED"] = redeemed_flag.reindex(per_hh.index)
    per_household_frames.append(per_hh)

behavior_all = pd.concat(per_household_frames)
behavior_all.head()

**Bước 3 - Gộp theo `REDEEMED` để so sánh 2 nhóm.**

In [ ]:
behavior_summary = behavior_all.groupby("REDEEMED").agg(
    n_household_campaign_pairs=("n_invoices", "count"),
    avg_n_invoices=("n_invoices", "mean"),
    avg_items_per_invoice=("avg_items_per_invoice", "mean"),
    avg_spend=("total_spend", "mean"),
).rename(index={True: "Redeemed", False: "Not redeemed"})

print("Behavior during campaign window, pooled across all 30 campaigns:")
display(behavior_summary.round(2))

### Insights: Output C

Household có redeem coupon (trong campaign đó) mua sắm nhiều hơn RÕ RỆT so với household không redeem,
trong chính lúc campaign diễn ra:
- Số invoice (trip) trung bình: khoảng 16.3 so với 13.4 (cao hơn khoảng 22%)
- Số món trung bình/invoice: khoảng 1599 so với 956 (cao hơn khoảng 67%)
- Chi tiêu trung bình: khoảng $623 so với $422 (cao hơn khoảng 48%)

-> Việc redeem coupon là một tín hiệu mạnh cho "khách hàng gắn bó" (engaged customer) - household dùng
coupon không chỉ mua nhiều hơn mỗi lần, mà còn đi mua thường xuyên hơn trong lúc campaign. Lưu ý đây là so
sánh TƯƠNG QUAN (correlation), không khẳng định việc dùng coupon là NGUYÊN NHÂN trực tiếp làm họ mua nhiều
hơn (có thể họ vốn đã là khách hàng trung thành/hay mua sắm, nên cũng dễ redeem coupon hơn).

## Tổng kết BQ2

- Tỷ lệ redeem khác nhau khá nhiều giữa các campaign (1.5% - 18.9%), không liên quan rõ ràng đến quy mô
  campaign.
- Người redeem coupon có xu hướng lớn tuổi hơn và thu nhập trung bình-khá cao hơn người không redeem, và
  đặc biệt là chủ nhà nhiều hơn hẳn.
- Người redeem chi tiêu nhiều hơn đáng kể trong lúc campaign về mọi mặt (số lần mua, số món/lần mua, tổng
  chi tiêu).